In [1]:
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install git+https://github.com/huggingface/transformers.git --upgrade
!pip install accelerate safetensors pandas

Looking in indexes: https://download.pytorch.org/whl/cu121
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-x9vqpo7s
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-x9vqpo7s
  Resolved https://github.com/huggingface/transformers.git to commit cac0a28c83cf87b7a05495de3177099c635ba852
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.1/516.1 kB 14.3 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.0.0.dev0-py3-none-any.whl size=10801169 sha256=5af80e616dc6d2a1378ce7dbc5e15fc2c9f591444923228e470ea58479abd3a8
  Stored in directory: /tmp/pip-ephem-wheel-cache-fvyv30oh/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installa

In [2]:
import google.colab.auth
google.colab.auth.authenticate_user()

# Local Inference & Comparison
This notebook downloads the fine-tuned adapters from GCS and runs inference locally using the `transformers` library.

In [3]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0))

CUDA Available: True
Device Name: Tesla T4


In [4]:
import transformers
print(transformers.__version__)

5.0.0.dev0


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.cloud import storage
import pandas as pd
import json
import os
import shutil
import gc
from sklearn.metrics import accuracy_score, classification_report

# --- CONFIGURATION ---
BUCKET_NAME = "mlops-22f3002292"

# REPLACE these with the actual output paths from your Vertex AI Tuning Job
# Example: "finetuning/output/v1_model/experiment_run/.../model"
GCS_MODEL_PATH_V1 = "week10_finetuning/output_v1/gemma-3-1b-it-1764480276573-20251130114119/merged_model"
GCS_MODEL_PATH_V2 = "week10_finetuning/output_v2/gemma-3-1b-it-1764484594678-20251130120750/merged_model"

LOCAL_DIR_V1 = "./models/v1"
LOCAL_DIR_V2 = "./models/v2"

print("CUDA Available:", torch.cuda.is_available())

CUDA Available: True


In [6]:
# Helper: Download Model from GCS
def download_model_from_gcs(gcs_path, local_dir):
    if os.path.exists(local_dir):
        print(f"Directory {local_dir} already exists. Skipping download.")
        return

    print(f"Downloading from gs://{BUCKET_NAME}/{gcs_path} to {local_dir}...")
    # Using gsutil for efficiency in notebook environment
    os.makedirs(local_dir, exist_ok=True)
    !gsutil -m -o GSUtil:check_hashes=never cp -r gs://{BUCKET_NAME}/{gcs_path}/* {local_dir}
    print("Download complete.")

# Helper: Load Test Data
def load_test_data(gcs_path):
    client = storage.Client()
    blob = client.bucket(BUCKET_NAME).blob(gcs_path)
    content = blob.download_as_text()
    data = []
    for line in content.strip().split('\n'):
        data.append(json.loads(line))
    return data

test_data_v1 = load_test_data("week10_finetuning/v1/test.jsonl")
test_data_v2 = load_test_data("week10_finetuning/v2/test.jsonl")

In [7]:
# Helper: Construct Prompt (Fixed to match training data)
def make_prompt(text):
    prompt = (
    "<start_of_turn>system\n"
    "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]\n"
    "<end_of_turn>\n"
    "<start_of_turn>user\n"
    +text+
    "<end_of_turn>\n"
    "<start_of_turn>assistant\n"
    )

    return prompt

# Verify the prompt structure matches your training data expectations
print(make_prompt(test_data_v1[0]['messages'][1]['content']))

<start_of_turn>system
Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]
<end_of_turn>
<start_of_turn>user
Sepal Length: 6.2, Sepal Width: 2.2, Petal Length: 4.5, Petal Width: 1.5<end_of_turn>
<start_of_turn>assistant



In [10]:
# Evaluation Function
def evaluate_local_model(model_local_path, test_data, model_name):
    print(f"\n=== Loading {model_name} from {model_local_path} ===")

    # Load Tokenizer & Model
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_local_path)
        model = AutoModelForCausalLM.from_pretrained(model_local_path, device_map="auto", use_safetensors=False)
    except Exception as e:
        print(f"FAILED to load model: {e}")
        return [], []

    preds = []
    trues = []

    print("Starting Inference...", end=" ")
    for i, entry in enumerate(test_data):
        prompt = make_prompt(entry['messages'][1]['content'])
        target = next(m['content'] for m in entry['messages'] if m['role'] == 'assistant')

        inputs = tokenizer(prompt, return_tensors="pt").to('cuda')

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=2,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode new tokens only
        generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        cleaned_pred = generated_text.strip()

        # Debug first prediction to verify fix
        if i == 0:
            print(f"\n[DEBUG] Prompt: {prompt!r}")
            print(f"[DEBUG] Raw Output: {generated_text!r}")

        preds.append(cleaned_pred)
        trues.append(target.strip())

        if i % 10 == 0: print(".", end="")

    print("\nDone.")

    # Memory Cleanup
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    print("🧹 Memory cleared.")

    return trues, preds

In [11]:
# --- 1. Evaluate V1 (Raw) ---
download_model_from_gcs(GCS_MODEL_PATH_V1, LOCAL_DIR_V1)
y_true_v1, y_pred_v1 = evaluate_local_model(LOCAL_DIR_V1, test_data_v1, "V1 (Raw)")

# # --- 2. Evaluate V2 (Descriptive) ---
download_model_from_gcs(GCS_MODEL_PATH_V2, LOCAL_DIR_V2)
y_true_v2, y_pred_v2 = evaluate_local_model(LOCAL_DIR_V2, test_data_v2, "V2 (Descriptive)")

Directory ./models/v1 already exists. Skipping download.

=== Loading V1 (Raw) from ./models/v1 ===


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

Starting Inference... 
[DEBUG] Prompt: '<start_of_turn>system\nClassify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]\n<end_of_turn>\n<start_of_turn>user\nSepal Length: 6.2, Sepal Width: 2.2, Petal Length: 4.5, Petal Width: 1.5<end_of_turn>\n<start_of_turn>assistant\n'
[DEBUG] Raw Output: 'versicolor'
....
Done.
🧹 Memory cleared.
Directory ./models/v2 already exists. Skipping download.

=== Loading V2 (Descriptive) from ./models/v2 ===


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

Starting Inference... 
[DEBUG] Prompt: '<start_of_turn>system\nClassify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]\n<end_of_turn>\n<start_of_turn>user\nSepal Length is Medium, Sepal Width is Low, Petal Length is Medium, Petal Width is Medium<end_of_turn>\n<start_of_turn>assistant\n'
[DEBUG] Raw Output: 'versicolor'
....
Done.
🧹 Memory cleared.


In [12]:
def show_results(title, trues, preds):
    print(f"\n=== {title} ===")
    acc = accuracy_score(trues, preds)
    print(f"Accuracy: {acc:.2%}")
    # We force valid labels to avoid errors if model hallucinates
    print(classification_report(trues, preds, labels=['setosa', 'versicolor', 'virginica'], zero_division=0))

show_results("V1 Results", y_true_v1, y_pred_v1)
show_results("V2 Results", y_true_v2, y_pred_v2)

# Comparison Logic
acc1 = accuracy_score(y_true_v1, y_pred_v1)
acc2 = accuracy_score(y_true_v2, y_pred_v2)

print("\n--- Final Verdict ---")
if acc2 > acc1:
    print(f"Descriptive data (V2) performed better by {(acc2-acc1)*100:.2f}%")
elif acc1 > acc2:
    print(f"Raw numeric data (V1) performed better by {(acc1-acc2)*100:.2f}%")
else:
    print("Both models performed equally.")


=== V1 Results ===
Accuracy: 32.26%
              precision    recall  f1-score   support

      setosa       0.00      0.00      0.00        10
  versicolor       0.32      1.00      0.49        10
   virginica       0.00      0.00      0.00        11

    accuracy                           0.32        31
   macro avg       0.11      0.33      0.16        31
weighted avg       0.10      0.32      0.16        31


=== V2 Results ===
Accuracy: 29.03%
              precision    recall  f1-score   support

      setosa       0.00      0.00      0.00        10
  versicolor       0.30      0.90      0.45        10
   virginica       0.00      0.00      0.00        11

   micro avg       0.30      0.29      0.30        31
   macro avg       0.10      0.30      0.15        31
weighted avg       0.10      0.29      0.15        31


--- Final Verdict ---
Raw numeric data (V1) performed better by 3.23%
